# 📊 Data Science Fundamentals: NumPy, Pandas & Visualization

**A hands-on tutorial notebook for aspiring data scientists.**

---

## What You'll Learn

| Section | Topics |
|---------|--------|
| **Part 1 – NumPy** | Array creation, indexing, broadcasting, vectorization, linear algebra |
| **Part 2 – Pandas** | DataFrames, cleaning, groupby, merging, pivot tables |
| **Part 3 – Visualization** | Matplotlib, Seaborn, subplots, heatmaps, dashboards |

> **Prerequisites:** Basic Python knowledge (variables, loops, functions).
> All data in this notebook is **synthetic** – no downloads required.

Let's get started! 🚀

---
# Part 1: NumPy Fundamentals

## What is NumPy?

**NumPy** (Numerical Python) is the foundational package for scientific computing in Python.
It provides:

- **`ndarray`** – a fast, memory-efficient N-dimensional array
- Element-wise operations and **broadcasting**
- Linear algebra, Fourier transforms, and random number generation
- The backbone of nearly every data science library (Pandas, Scikit-learn, TensorFlow, …)

### Why does it matter?

Python lists are flexible but *slow* for numerical work. NumPy arrays store data in
contiguous memory and delegate computation to optimised C/Fortran routines, delivering
**10–100× speed-ups** over pure Python loops.

> 💡 **Tip:** Think of NumPy as the "engine" under the hood of the Python data science stack.

## 1.1 Array Creation

NumPy offers many ways to create arrays. The most common are shown below.

In [ ]:
import numpy as np

# From a Python list
a = np.array([1, 2, 3, 4, 5])
print("From list:      ", a, " | dtype:", a.dtype)

# Pre-filled arrays
print("Zeros (2x3):\n", np.zeros((2, 3)))
print("Ones  (3,):  ", np.ones(3))
print("Full  (2x2):\n", np.full((2, 2), 7))

# Sequences
print("arange(0,10,2):", np.arange(0, 10, 2))
print("linspace(0,1,5):", np.linspace(0, 1, 5))

# Random arrays (using the modern Generator API)
rng = np.random.default_rng(42)
print("Random ints:   ", rng.integers(0, 100, size=5))
print("Random floats: ", rng.random(5).round(3))
print("Normal dist:   ", rng.normal(loc=0, scale=1, size=5).round(3))

## 1.2 Indexing & Slicing

NumPy indexing works like Python lists but extends to multiple dimensions.

| Syntax | Meaning |
|--------|---------|
| `a[2]` | Single element |
| `a[1:4]` | Slice (start:stop) |
| `a[::2]` | Every 2nd element |
| `M[row, col]` | 2-D element |
| `M[:, 0]` | Entire first column |

> ⚠️ **Warning:** NumPy slices return *views*, not copies. Modifying a slice modifies the original array.

In [ ]:
# 1-D indexing
arr = np.arange(10)
print("Array:        ", arr)
print("arr[3]:       ", arr[3])
print("arr[2:7]:     ", arr[2:7])
print("arr[::3]:     ", arr[::3])       # every 3rd element
print("arr[-3:]:     ", arr[-3:])       # last 3 elements

# 2-D indexing
matrix = np.arange(12).reshape(3, 4)
print("\nMatrix:\n", matrix)
print("Row 1:        ", matrix[1])
print("Element (2,3):", matrix[2, 3])
print("Column 0:     ", matrix[:, 0])
print("Sub-matrix:\n", matrix[:2, 1:3])  # first 2 rows, cols 1-2

# Boolean (fancy) indexing
print("\nElements > 5: ", matrix[matrix > 5])

## 1.3 Array Operations & Broadcasting

### Element-wise operations
Arithmetic operators (`+`, `-`, `*`, `/`, `**`) work **element-wise** on arrays of the same shape.

### Broadcasting rules
When shapes differ, NumPy tries to "broadcast" the smaller array:

1. If the arrays differ in number of dimensions, the shape of the smaller array is padded with 1s on the left.
2. Arrays with size 1 along a dimension act as if they had the size of the largest array in that dimension.
3. If sizes disagree and neither is 1 → **error**.

```
(3, 4) + (4,)   →  OK  (row broadcast)
(3, 1) + (1, 4) →  OK  (both broadcast → 3×4)
(3, 4) + (3,)   →  ERROR
```

In [ ]:
# Element-wise arithmetic
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])
print("a + b  =", a + b)
print("a * b  =", a * b)
print("a ** 2 =", a ** 2)

# Broadcasting: scalar
print("\na + 100 =", a + 100)

# Broadcasting: row vector + column vector
row = np.array([[1, 2, 3]])        # shape (1, 3)
col = np.array([[10], [20], [30]]) # shape (3, 1)
result = row + col                  # shape (3, 3)
print("\nRow (1,3) + Col (3,1) =\n", result)

# Practical: normalise each column to 0-1 range
data = rng.integers(0, 100, size=(4, 3)).astype(float)
col_min = data.min(axis=0)   # shape (3,)
col_max = data.max(axis=0)
normalised = (data - col_min) / (col_max - col_min)
print("\nOriginal:\n", data)
print("Normalised:\n", normalised.round(2))

## 1.4 Vectorized Operations vs Loops

One of NumPy's biggest advantages is **vectorization**: replacing explicit Python loops
with array expressions that run in compiled C code.

Below we compare computing the sum of squares for 1 million numbers using a Python loop
vs a NumPy one-liner.

In [ ]:
import time

size = 1_000_000
data = rng.random(size)

# --- Python loop ---
start = time.perf_counter()
total_loop = 0.0
for x in data:
    total_loop += x * x
loop_time = time.perf_counter() - start

# --- NumPy vectorized ---
start = time.perf_counter()
total_np = np.sum(data ** 2)
np_time = time.perf_counter() - start

print(f"Python loop : {loop_time:.4f}s  (result={total_loop:.2f})")
print(f"NumPy vector: {np_time:.6f}s  (result={total_np:.2f})")
print(f"Speed-up    : {loop_time / np_time:.0f}×")

## 1.5 Linear Algebra

NumPy's `linalg` sub-module provides all the essentials:

| Function | Description |
|----------|-------------|
| `np.dot(a, b)` | Dot product (1-D) or matrix multiply |
| `a @ b` | Matrix multiplication operator |
| `np.linalg.det(A)` | Determinant |
| `np.linalg.inv(A)` | Inverse |
| `np.linalg.eig(A)` | Eigenvalues & eigenvectors |
| `np.linalg.solve(A, b)` | Solve Ax = b |

In [ ]:
# Dot product
v1 = np.array([1, 2, 3])
v2 = np.array([4, 5, 6])
print("Dot product:", np.dot(v1, v2))   # 1*4 + 2*5 + 3*6 = 32

# Matrix multiplication
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print("\nA @ B =\n", A @ B)

# Determinant & inverse
print("\ndet(A) =", np.linalg.det(A))
A_inv = np.linalg.inv(A)
print("inv(A) =\n", A_inv)
print("A @ inv(A) ≈ I:\n", (A @ A_inv).round(10))

# Eigenvalues & eigenvectors
sym = np.array([[4, 2], [2, 3]])
eigenvalues, eigenvectors = np.linalg.eig(sym)
print("\nEigenvalues: ", eigenvalues)
print("Eigenvectors:\n", eigenvectors)

# Solve Ax = b
b = np.array([1, 2])
x = np.linalg.solve(A, b)
print("\nSolve Ax=b → x =", x)
print("Verify A@x  =", A @ x)

## 1.6 Useful Array Functions

A quick reference for functions you'll use constantly.

In [ ]:
arr = np.array([3, 1, 4, 1, 5, 9, 2, 6])

# Reshape
print("Reshape (2,4):\n", arr.reshape(2, 4))

# Concatenate
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print("Concatenate:", np.concatenate([a, b]))
print("Stack rows:\n", np.vstack([a, b]))
print("Stack cols:\n", np.column_stack([a, b]))

# Where (conditional)
print("\nWhere > 3:  ", np.where(arr > 3, arr, 0))   # keep if >3, else 0

# Argmax / Argmin
print("argmax:     ", np.argmax(arr), "→ value", arr[np.argmax(arr)])
print("argmin:     ", np.argmin(arr), "→ value", arr[np.argmin(arr)])

# Sort
print("Sorted:     ", np.sort(arr))
print("Sort index: ", np.argsort(arr))

# Unique
print("Unique:     ", np.unique(arr))

---
# Part 2: Pandas Fundamentals

## What is Pandas?

**Pandas** is the primary data-manipulation library in Python, built on top of NumPy.
It introduces two key data structures:

| Structure | Description | Analogy |
|-----------|-------------|---------|
| **Series** | 1-D labelled array | A single spreadsheet column |
| **DataFrame** | 2-D labelled table | An entire spreadsheet / SQL table |

### Why Pandas?
- **Labelled axes** – reference data by name, not just position
- **Automatic alignment** – operations align on index/columns
- **Missing data** – first-class `NaN` support
- **Rich I/O** – read/write CSV, Excel, SQL, Parquet, …
- **GroupBy** – split-apply-combine in one line

> 💡 **Tip:** Pandas ≥ 2.0 uses PyArrow-backed types by default for many operations,
> giving better performance and nullable integer/string support.

## 2.1 Creating a Synthetic Sales Dataset

We'll generate a realistic dataset with **1 000 rows** of daily sales transactions.
This avoids any need to download files and lets us control the data characteristics.

In [ ]:
import pandas as pd

rng = np.random.default_rng(42)
n = 1000

# Date range: one year of business days, sampled randomly
dates = pd.bdate_range("2024-01-01", periods=252)  # ~1 yr business days
date_col = rng.choice(dates, size=n)

# Products & categories
products = {
    "Widget A": "Gadgets",
    "Widget B": "Gadgets",
    "Gizmo X":  "Electronics",
    "Gizmo Y":  "Electronics",
    "Doohickey": "Accessories",
    "Thingamajig": "Accessories",
}
product_names = list(products.keys())
product_col = rng.choice(product_names, size=n)
category_col = [products[p] for p in product_col]

# Prices per product (fixed)
price_map = {
    "Widget A": 19.99, "Widget B": 29.99,
    "Gizmo X": 49.99,  "Gizmo Y": 99.99,
    "Doohickey": 9.99,  "Thingamajig": 14.99,
}
price_col = np.array([price_map[p] for p in product_col])

# Quantities (1-20, right-skewed)
quantity_col = rng.integers(1, 21, size=n)

# Regions & customers
regions = ["North", "South", "East", "West"]
region_col = rng.choice(regions, size=n)
customer_ids = rng.integers(1000, 2000, size=n)

# Build DataFrame
sales = pd.DataFrame({
    "date":        date_col,
    "product":     product_col,
    "category":    category_col,
    "quantity":    quantity_col,
    "unit_price":  price_col,
    "revenue":     (quantity_col * price_col).round(2),
    "region":      region_col,
    "customer_id": customer_ids,
})
sales = sales.sort_values("date").reset_index(drop=True)
print(f"Dataset shape: {sales.shape}")
sales.head(10)

## 2.2 Basic Exploration

The first thing to do with any new dataset: **look at it**.

| Method | What it tells you |
|--------|-------------------|
| `.head()` / `.tail()` | First/last rows |
| `.shape` | (rows, columns) |
| `.dtypes` | Column data types |
| `.info()` | Non-null counts + dtypes + memory |
| `.describe()` | Summary statistics for numeric columns |

In [ ]:
print("Shape:", sales.shape)
print("\nData types:\n", sales.dtypes)
print("\n--- .info() ---")
sales.info()
print("\n--- .describe() ---")
sales.describe()

## 2.3 Indexing & Filtering

Pandas provides several ways to select data:

| Accessor | Index by |
|----------|----------|
| `df["col"]` | Column name → Series |
| `df.loc[row_label, col_label]` | **Label**-based |
| `df.iloc[row_pos, col_pos]` | **Position**-based (integer) |
| `df[bool_series]` | Boolean mask |

> ⚠️ **Warning:** `df.loc` includes *both* endpoints; `df.iloc` excludes the stop index (like Python slicing).

In [ ]:
# Single column → Series
print(sales["product"].head())

# .loc – label-based (inclusive)
print("\nloc[0:2, 'product':'quantity']:\n",
      sales.loc[0:2, "product":"quantity"])

# .iloc – position-based (exclusive stop)
print("\niloc[0:3, 0:4]:\n", sales.iloc[0:3, 0:4])

# Boolean filtering – high-revenue electronics
mask = (sales["category"] == "Electronics") & (sales["revenue"] > 500)
print(f"\nHigh-revenue Electronics rows: {mask.sum()}")
print(sales.loc[mask, ["date", "product", "quantity", "revenue"]].head())

## 2.4 Handling Missing Values

Real datasets are messy. Let's introduce some `NaN` values and practise cleaning them.

### Strategy guide:
- **Drop** rows/columns if missingness is low and data is plentiful
- **Fill** with mean/median/mode when appropriate
- **Forward/back fill** for time series
- **Flag** with an indicator column when missingness is informative

In [ ]:
# Make a copy and inject NaNs
df = sales.copy()
np.random.seed(0)
nan_idx = np.random.choice(df.index, size=50, replace=False)
df.loc[nan_idx[:25], "quantity"] = np.nan
df.loc[nan_idx[25:], "revenue"] = np.nan

# Inspect
print("Missing values per column:")
print(df.isna().sum())
print(f"\nTotal rows with any NaN: {df.isna().any(axis=1).sum()}")

# Fill quantity with median, revenue with 0
df["quantity"] = df["quantity"].fillna(df["quantity"].median())
df["revenue"]  = df["revenue"].fillna(0)
print("\nAfter filling:")
print(df.isna().sum())

# dropna example (on a small sample)
sample = sales.head(10).copy()
sample.iloc[2, 3] = np.nan
print("\nBefore dropna:\n", sample[["product", "quantity"]].to_string())
print("After dropna:\n", sample.dropna(subset=["quantity"])[["product", "quantity"]].to_string())

## 2.5 Data Transformation

### Key tools:
- **`.apply(func)`** – apply any function row-wise or column-wise
- **`.map(dict_or_func)`** – element-wise mapping on a Series
- **Type conversion** – `.astype()`, `pd.to_datetime()`
- **String methods** – `.str.lower()`, `.str.contains()`, etc.

In [ ]:
# .apply – create a discount tier based on quantity
def discount_tier(qty):
    if qty >= 15:
        return "Gold"
    elif qty >= 8:
        return "Silver"
    return "Bronze"

sales["tier"] = sales["quantity"].apply(discount_tier)
print(sales["tier"].value_counts())

# .map – short region names
region_short = {"North": "N", "South": "S", "East": "E", "West": "W"}
sales["region_code"] = sales["region"].map(region_short)

# String methods
sales["product_lower"] = sales["product"].str.lower()
gadgets = sales[sales["product"].str.contains("Widget")]
print(f"\nRows containing 'Widget': {len(gadgets)}")

# Type info
print("\nDate dtype:", sales["date"].dtype)
sales["month"] = sales["date"].dt.month
sales["day_name"] = sales["date"].dt.day_name()
print(sales[["date", "month", "day_name"]].head())

## 2.6 GroupBy & Aggregation

The **split-apply-combine** pattern is Pandas' super-power:

1. **Split** the data into groups
2. **Apply** a function to each group
3. **Combine** results into a new DataFrame

```python
df.groupby("col").agg(func)
```

In [ ]:
# Revenue by category
print("=== Revenue by Category ===")
cat_stats = sales.groupby("category")["revenue"].agg(["sum", "mean", "count"])
print(cat_stats.round(2))

# Multiple aggregations
print("\n=== By Region & Category ===")
multi = (sales
    .groupby(["region", "category"])
    .agg(
        total_revenue=("revenue", "sum"),
        avg_quantity=("quantity", "mean"),
        num_orders=("revenue", "count"),
    )
    .round(2)
)
print(multi)

# Top product by total revenue
print("\n=== Top 3 Products by Revenue ===")
top = (sales.groupby("product")["revenue"]
       .sum()
       .sort_values(ascending=False)
       .head(3))
print(top)

## 2.7 Merging & Concatenation

### `pd.merge()` – SQL-style joins
- `how`: `'inner'`, `'left'`, `'right'`, `'outer'`
- `on`: column(s) to join on

### `pd.concat()` – stacking DataFrames
- `axis=0`: stack rows (vertical)
- `axis=1`: stack columns (horizontal)

In [ ]:
# Product info table
product_info = pd.DataFrame({
    "product": list(price_map.keys()),
    "weight_kg": [0.3, 0.5, 1.2, 2.0, 0.1, 0.15],
    "supplier":  ["SupA", "SupA", "SupB", "SupB", "SupC", "SupC"],
})
print("Product Info:\n", product_info, "\n")

# Inner merge
merged = sales.merge(product_info, on="product", how="left")
print("Merged shape:", merged.shape)
print(merged[["product", "revenue", "weight_kg", "supplier"]].head())

# Concat – stack two months
jan = sales[sales["month"] == 1]
feb = sales[sales["month"] == 2]
q1_partial = pd.concat([jan, feb], ignore_index=True)
print(f"\nJan rows: {len(jan)}, Feb rows: {len(feb)}, Combined: {len(q1_partial)}")

## 2.8 Pivot Tables & Crosstabs

Pivot tables reshape data for analysis — think Excel PivotTables.

```python
pd.pivot_table(df, values=..., index=..., columns=..., aggfunc=...)
pd.crosstab(df["col1"], df["col2"])
```

In [ ]:
# Pivot table: avg revenue by region × category
pivot = pd.pivot_table(
    sales,
    values="revenue",
    index="region",
    columns="category",
    aggfunc="mean",
).round(2)
print("Pivot – Mean Revenue:\n", pivot)

# Crosstab: count of orders by region × tier
ct = pd.crosstab(sales["region"], sales["tier"], margins=True)
print("\nCrosstab – Order Counts:\n", ct)

---
# Part 3: Data Visualization

## Matplotlib vs Seaborn

| | Matplotlib | Seaborn |
|---|---|---|
| **Level** | Low-level | High-level (built on Matplotlib) |
| **Strength** | Full control over every element | Beautiful statistical plots with minimal code |
| **When to use** | Custom layouts, animations, fine-tuning | EDA, statistical summaries, publication-ready plots |

Both are essential tools. Matplotlib gives you the canvas; Seaborn gives you smart defaults.

> 💡 **Tip:** Seaborn functions return Matplotlib `Axes` objects, so you can always
> fall back to Matplotlib for fine-tuning.

## 3.1 Matplotlib Basics

The fundamental pattern:
```python
fig, ax = plt.subplots()
ax.plot(x, y)
ax.set_title(...)
plt.show()
```

In [ ]:
import matplotlib
matplotlib.use("Agg")          # non-interactive backend for script execution
import matplotlib.pyplot as plt

# --- Line plot: monthly revenue trend ---
monthly_rev = sales.groupby("month")["revenue"].sum()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(monthly_rev.index, monthly_rev.values, marker="o", color="steelblue", linewidth=2)
ax.set_title("Monthly Revenue Trend (2024)", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Total Revenue ($)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plot_line.png", dpi=100)
plt.close()
print("✅ Line plot saved to plot_line.png")

In [ ]:
# --- Bar chart: revenue by product ---
prod_rev = sales.groupby("product")["revenue"].sum().sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(prod_rev)))
ax.barh(prod_rev.index, prod_rev.values, color=colors)
ax.set_title("Total Revenue by Product", fontsize=14)
ax.set_xlabel("Revenue ($)")
for i, v in enumerate(prod_rev.values):
    ax.text(v + 500, i, f"${v:,.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("plot_bar.png", dpi=100)
plt.close()
print("✅ Bar chart saved to plot_bar.png")

In [ ]:
# --- Scatter plot: quantity vs revenue ---
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    sales["quantity"], sales["revenue"],
    c=sales["category"].astype("category").cat.codes,
    cmap="Set2", alpha=0.6, edgecolors="w", linewidth=0.5
)
ax.set_title("Quantity vs Revenue", fontsize=14)
ax.set_xlabel("Quantity Sold")
ax.set_ylabel("Revenue ($)")
# Legend
handles, _ = scatter.legend_elements()
ax.legend(handles, sales["category"].unique(), title="Category")
plt.tight_layout()
plt.savefig("plot_scatter.png", dpi=100)
plt.close()
print("✅ Scatter plot saved to plot_scatter.png")

## 3.2 Subplots – Multiple Charts in One Figure

Use `plt.subplots(nrows, ncols)` to create a grid of axes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# (0,0) Histogram of revenue
axes[0, 0].hist(sales["revenue"], bins=30, color="skyblue", edgecolor="white")
axes[0, 0].set_title("Revenue Distribution")
axes[0, 0].set_xlabel("Revenue ($)")

# (0,1) Box plot of quantity by region
region_data = [sales[sales["region"] == r]["quantity"].values for r in regions]
bp = axes[0, 1].boxplot(region_data, tick_labels=regions, patch_artist=True)
for patch, color in zip(bp["boxes"], ["#ff9999", "#66b3ff", "#99ff99", "#ffcc99"]):
    patch.set_facecolor(color)
axes[0, 1].set_title("Quantity by Region")

# (1,0) Pie chart of category share
cat_counts = sales["category"].value_counts()
axes[1, 0].pie(cat_counts, labels=cat_counts.index, autopct="%1.1f%%",
               startangle=90, colors=["#ff9999", "#66b3ff", "#99ff99"])
axes[1, 0].set_title("Sales by Category")

# (1,1) Stacked bar: region × category revenue
pivot_data = sales.pivot_table(values="revenue", index="region",
                                columns="category", aggfunc="sum")
pivot_data.plot(kind="bar", stacked=True, ax=axes[1, 1], colormap="Pastel1")
axes[1, 1].set_title("Revenue: Region × Category")
axes[1, 1].set_xlabel("")
axes[1, 1].legend(fontsize=8)
axes[1, 1].tick_params(axis="x", rotation=0)

fig.suptitle("Sales Overview Dashboard", fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig("plot_subplots.png", dpi=100, bbox_inches="tight")
plt.close()
print("✅ Subplots figure saved to plot_subplots.png")

## 3.3 Seaborn – Statistical Visualization

Seaborn wraps Matplotlib with smart defaults and integrates directly with DataFrames.

In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")

# histplot – distribution of revenue
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=sales, x="revenue", hue="category", bins=30,
             kde=True, alpha=0.5, ax=ax)
ax.set_title("Revenue Distribution by Category")
plt.tight_layout()
plt.savefig("plot_seaborn_hist.png", dpi=100)
plt.close()
print("✅ Seaborn histplot saved")

In [ ]:
# boxplot + violinplot side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    warnings.simplefilter("ignore", PendingDeprecationWarning)
    sns.boxplot(data=sales, x="category", y="revenue", hue="category",
                palette="Set2", legend=False, ax=ax1)
ax1.set_title("Box Plot: Revenue by Category")

with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    warnings.simplefilter("ignore", PendingDeprecationWarning)
    sns.violinplot(data=sales, x="region", y="quantity", hue="region",
                   palette="muted", legend=False, inner="quart", ax=ax2)
ax2.set_title("Violin Plot: Quantity by Region")

plt.tight_layout()
plt.savefig("plot_box_violin.png", dpi=100)
plt.close()
print("✅ Box + Violin plots saved")

In [ ]:
# pairplot on a subset of numeric columns
subset = sales[["quantity", "unit_price", "revenue", "category"]].copy()
g = sns.pairplot(subset, hue="category", palette="husl",
                 diag_kind="kde", height=2.2, plot_kws={"alpha": 0.5})
g.figure.suptitle("Pairplot of Sales Metrics", y=1.02, fontsize=14)
g.figure.savefig("plot_pairplot.png", dpi=100, bbox_inches="tight")
plt.close("all")
print("✅ Pairplot saved")

## 3.4 Heatmap – Correlation Matrix

A heatmap of the correlation matrix reveals linear relationships between numeric variables.

In [ ]:
# Correlation matrix
numeric_cols = sales.select_dtypes(include="number")
corr = numeric_cols.corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig("plot_heatmap.png", dpi=100)
plt.close()
print("✅ Heatmap saved")
print("\nCorrelation matrix:\n", corr.round(2))

---
## 🏋️ Exercise: Build a Sales Dashboard

**Your task:** Create a single figure with **4 subplots** that tells a story about the sales data.

### Requirements:
1. **Top-left:** A line chart of *weekly* revenue (resample the data by week)
2. **Top-right:** A grouped bar chart comparing mean revenue across regions for each category
3. **Bottom-left:** A box plot of revenue by day of the week (Mon–Fri)
4. **Bottom-right:** A heatmap showing average quantity sold per product × region

### Starter code is provided below – fill in the `# TODO` sections!

> 💡 **Hint:** Use `sales.set_index("date").resample("W")["revenue"].sum()` for weekly revenue.

In [ ]:
# ============================================================
# 🏋️  EXERCISE – Fill in the TODOs
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- 1. Weekly revenue trend ---
# TODO: resample sales by week, plot on axes[0, 0]
# weekly = sales.set_index("date").resample("W")["revenue"].sum()
# axes[0, 0].plot(...)
axes[0, 0].set_title("Weekly Revenue Trend")

# --- 2. Grouped bar: mean revenue by region × category ---
# TODO: use pivot_table and .plot(kind="bar") on axes[0, 1]
# pivot = pd.pivot_table(sales, values="revenue", index="region",
#                         columns="category", aggfunc="mean")
# pivot.plot(kind="bar", ax=axes[0, 1])
axes[0, 1].set_title("Mean Revenue: Region × Category")

# --- 3. Box plot: revenue by day of week ---
# TODO: use sns.boxplot on axes[1, 0]
# day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
# sns.boxplot(data=sales, x="day_name", y="revenue",
#             order=day_order, ax=axes[1, 0])
axes[1, 0].set_title("Revenue by Day of Week")

# --- 4. Heatmap: avg quantity per product × region ---
# TODO: create pivot_table & use sns.heatmap on axes[1, 1]
# heat_data = pd.pivot_table(sales, values="quantity",
#                             index="product", columns="region", aggfunc="mean")
# sns.heatmap(heat_data, annot=True, fmt=".1f", cmap="YlOrRd", ax=axes[1, 1])
axes[1, 1].set_title("Avg Quantity: Product × Region")

fig.suptitle("Sales Dashboard Exercise", fontsize=16, y=1.01)
plt.tight_layout()
# plt.savefig("exercise_dashboard.png", dpi=100, bbox_inches="tight")
plt.close()
print("TODO: Uncomment the code above and complete the exercise!")

---
# 🎓 Congratulations!

You've covered the three pillars of data science in Python:

| Pillar | Key Takeaway |
|--------|-------------|
| **NumPy** | Fast numerical computing with N-dimensional arrays |
| **Pandas** | Powerful tabular data manipulation & analysis |
| **Visualization** | Matplotlib for control, Seaborn for elegance |

## Next Steps

1. **Machine Learning Fundamentals** – Scikit-learn: regression, classification, clustering
2. **Advanced Pandas** – Window functions, `pipe()`, multi-index, categorical data
3. **Interactive Viz** – Plotly, Altair, Bokeh for dashboards
4. **Real Datasets** – Kaggle, UCI ML Repository, government open data portals
5. **Statistics** – Hypothesis testing, confidence intervals, Bayesian thinking

> *"Data is the new oil, but like oil, it's only valuable when refined."*

Happy exploring! 🚀